# Database Design Lab: Entscheidungen, die Jahre nachwirken

Dieses interaktive Notebook ist das Begleitmaterial zum Golem+ Artikel über Datenbank-Design in der KI-Ära.
Es demonstriert mechanisch, warum naives Design ("alles als VARCHAR") die Datenbank-Engine quält und wie bewusste Datentypen, Constraints und physische Ordnung (Zone Map Pruning) die Performance drastisch verbessern.

## Setup

In [1]:
!pip install duckdb -q

# Imports
import duckdb
import json
from pathlib import Path
from time import perf_counter

# Datenbank-Datei
DB_FILE = "db_design_lab.duckdb"
con = duckdb.connect(DB_FILE)

# Profiling aktivieren (optional, für detaillierte Analyse)
con.execute("PRAGMA enable_profiling='json';")
con.execute("PRAGMA profiling_output='profile.json';")
con.execute("PRAGMA profiling_mode='detailed';")

# Hilfsfunktion für Zeitmessung
def run_timed(sql, params=None):
    t0 = perf_counter()
    res = con.execute(sql, params or []).fetchall()
    dt_ms = (perf_counter() - t0) * 1000
    return dt_ms, res

print("Setup abgeschlossen.")

Setup abgeschlossen.


## Schritt 1 & 2: Das Naive Schema und Datengenerierung
Wir erstellen ein Schema, in dem aus Bequemlichkeit alles als Text (`VARCHAR`) gespeichert wird. Anschließend füllen wir es mit synthetischen Daten:
- 200 Firmen
- 20.000 Kandidaten
- 50.000 Stellenanzeigen
- **3.000.000 Bewerbungen**

In [2]:
# Bestehende Tabellen löschen (falls vorhanden)
con.execute("DROP TABLE IF EXISTS naive_companies;")
con.execute("DROP TABLE IF EXISTS naive_candidates;")
con.execute("DROP TABLE IF EXISTS naive_postings;")
con.execute("DROP TABLE IF EXISTS naive_apps;")

# Naives Schema: Strings für IDs und Zeit, wenig Regeln
con.execute("""
CREATE TABLE naive_companies (
    company_id VARCHAR,
    company_name VARCHAR,
    industry VARCHAR,
    created_at VARCHAR
);
""")

con.execute("""
CREATE TABLE naive_candidates (
    candidate_id VARCHAR,
    full_name VARCHAR,
    email VARCHAR,
    created_at VARCHAR
);
""")

con.execute("""
CREATE TABLE naive_postings (
    posting_id VARCHAR,
    company_id VARCHAR,
    title VARCHAR,
    location VARCHAR,
    posted_at VARCHAR,
    expires_at VARCHAR,
    status VARCHAR,
    created_at VARCHAR
);
""")

con.execute("""
CREATE TABLE naive_apps (
    application_id VARCHAR,
    posting_id VARCHAR,
    candidate_id VARCHAR,
    applied_at VARCHAR,
    stage VARCHAR,
    created_at VARCHAR
);
""")

print("Naives Schema erstellt.")

Naives Schema erstellt.


In [3]:
# Firmen (200)
con.execute("""
INSERT INTO naive_companies
SELECT
    i::VARCHAR,
    'Company ' || i::VARCHAR,
    CASE (i % 6)
        WHEN 0 THEN 'SaaS'
        WHEN 1 THEN 'FinTech'
        WHEN 2 THEN 'Health'
        WHEN 3 THEN 'E-Commerce'
        WHEN 4 THEN 'Manufacturing'
        ELSE 'Media'
    END,
    CAST(CURRENT_TIMESTAMP AS VARCHAR)
FROM range(1, 201) t(i);
""")

# Kandidaten (20.000)
con.execute("""
INSERT INTO naive_candidates
SELECT
    i::VARCHAR,
    'Candidate ' || i::VARCHAR,
    'candidate' || i::VARCHAR || '@example.com',
    CAST(CURRENT_TIMESTAMP AS VARCHAR)
FROM range(1, 20001) t(i);
""")

# Stellenanzeigen (50.000)
con.execute("""
INSERT INTO naive_postings
SELECT
    i::VARCHAR,
    (1 + (i % 200))::VARCHAR,
    CASE (i % 5)
        WHEN 0 THEN 'Data Analyst'
        WHEN 1 THEN 'Backend Engineer'
        WHEN 2 THEN 'ML Engineer'
        WHEN 3 THEN 'Product Manager'
        ELSE 'QA Engineer'
    END,
    CASE (i % 6)
        WHEN 0 THEN 'Berlin'
        WHEN 1 THEN 'Hamburg'
        WHEN 2 THEN 'M München'
        WHEN 3 THEN 'Remote'
        WHEN 4 THEN 'Köłłłn'
        ELSE 'Stuttgart'
    END,
    CAST((CURRENT_TIMESTAMP - INTERVAL '180 days' + (i % 180) * INTERVAL '1 day') AS VARCHAR),
    CAST((CURRENT_TIMESTAMP - INTERVAL '180 days' + (i % 180) * INTERVAL '1 day' + (7 + (i % 60)) * INTERVAL '1 day') AS VARCHAR),
    CASE
        WHEN (i % 100) < 10 THEN 'draft'
        WHEN (i % 100) < 80 THEN 'published'
        WHEN (i % 100) < 90 THEN 'filled'
        ELSE 'expired'
    END,
    CAST(CURRENT_TIMESTAMP AS VARCHAR)
FROM range(1, 50001) t(i);
""")

# Auszug: Bewerbungen (3.000.000)
con.execute("""
INSERT INTO naive_apps
SELECT
 i::VARCHAR,
 (1 + (i % 50000))::VARCHAR,
 (1 + ((i * 17) % 20000))::VARCHAR,
 CAST((CURRENT_TIMESTAMP - INTERVAL '180 days' + (i % 180) * INTERVAL '1 day' + (i % 1440) * INTERVAL '1 minute') AS VARCHAR),
 CASE (i % 6)
 WHEN 0 THEN 'submitted'
 WHEN 1 THEN 'screening'
 WHEN 2 THEN 'interview'
 WHEN 3 THEN 'offer'
 WHEN 4 THEN 'rejected'
 ELSE 'withdrawn'
 END,
 CAST(CURRENT_TIMESTAMP AS VARCHAR)
FROM range(1, 3000001) t(i);
""")
print("3 Millionen Bewerbungen eingefügt (naiv).")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

3 Millionen Bewerbungen eingefügt (naiv).


## Schritt 3: Die Beweis-Abfrage (Naiv)
Wir suchen nun alle veröffentlichten Anzeigen für Firma '42' und zählen die Bewerbungen. Da die `company_id` als `VARCHAR` vorliegt und die Tabelle unsortiert ist, muss DuckDB einen "Seq Scan" (Full Table Scan) machen und Millionen Zeilen lesen und konvertieren.
Wir nutzen `EXPLAIN ANALYZE`, um unter die Haube zu schauen.

In [4]:
q_naive = """
EXPLAIN ANALYZE
SELECT
    p.posting_id,
    p.status,
    COUNT(a.application_id) AS application_count
FROM naive_postings p
LEFT JOIN naive_apps a ON a.posting_id = p.posting_id
WHERE p.company_id = '42' AND p.status = 'published'
GROUP BY p.posting_id, p.status;
"""
dt_ms, plan = run_timed(q_naive)
print(f"Naiv Dauer: {dt_ms:,.2f} ms")
print(plan[0][0]) # Druckt den Abfrageplan

Naiv Dauer: 184.71 ms
analyzed_plan


## Schritt 4 & 5: Das Designed Schema (Typen, Constraints, Physische Ordnung)
Wir bauen das Schema jetzt nach professionellen Standards um:
1. **Ganzzahlen (BIGINT)** für IDs, damit Joins rasend schnell werden.
2. **TIMESTAMPS** statt Strings für effiziente Filter.
3. **CHECK-Constraints**, um KI-Halluzinationen und Müll-Daten auszuschließen.
4. **ORDER BY beim Insert:** Wir zwingen DuckDB, die Daten sortiert (nach `company_id` und `posted_at`) auf die Festplatte zu schreiben. Das aktiviert die internen Min/Max-Metadaten.

## Schritt 4: Designed Schema bauen


In [5]:
# Stage-Tabellen löschen
for t in ["stage_companies", "stage_candidates", "stage_postings", "stage_apps",
          "companies", "candidates", "job_postings", "applications",
          "posting_application_summary", "posting_notes"]:
    con.execute(f"DROP TABLE IF EXISTS {t};")

# Stage-Tabellen mit passenden Typen
con.execute("""
CREATE TABLE stage_companies (
    company_id BIGINT,
    company_name VARCHAR,
    industry VARCHAR,
    created_at TIMESTAMP
);
""")

con.execute("""
CREATE TABLE stage_candidates (
    candidate_id BIGINT,
    full_name VARCHAR,
    email VARCHAR,
    created_at TIMESTAMP
);
""")

con.execute("""
CREATE TABLE stage_postings (
    posting_id BIGINT,
    company_id BIGINT,
    title VARCHAR,
    location VARCHAR,
    posted_at TIMESTAMP,
    expires_at TIMESTAMP,
    status VARCHAR,
    created_at TIMESTAMP
);
""")

con.execute("""
CREATE TABLE stage_apps (
    application_id BIGINT,
    posting_id BIGINT,
    candidate_id BIGINT,
    applied_at TIMESTAMP,
    stage VARCHAR,
    created_at TIMESTAMP
);
""")

# Daten aus naiven Tabellen in Stage-Tabellen konvertieren
con.execute("""
INSERT INTO stage_companies
SELECT
    company_id::BIGINT,
    company_name,
    industry,
    created_at::TIMESTAMP
FROM naive_companies;
""")

con.execute("""
INSERT INTO stage_candidates
SELECT
    candidate_id::BIGINT,
    full_name,
    email,
    created_at::TIMESTAMP
FROM naive_candidates;
""")

con.execute("""
INSERT INTO stage_postings
SELECT
    posting_id::BIGINT,
    company_id::BIGINT,
    title,
    location,
    posted_at::TIMESTAMP,
    expires_at::TIMESTAMP,
    status,
    created_at::TIMESTAMP
FROM naive_postings;
""")

con.execute("""
INSERT INTO stage_apps
SELECT
    application_id::BIGINT,
    posting_id::BIGINT,
    candidate_id::BIGINT,
    applied_at::TIMESTAMP,
    stage,
    created_at::TIMESTAMP
FROM naive_apps;
""")

print("Stage-Tabellen erstellt und Daten konvertiert.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Stage-Tabellen erstellt und Daten konvertiert.


##Schritt 5: Final-Tabellen mit Regeln und Ordnung

In [6]:
# Final-Tabellen mit Constraints
con.execute("""
CREATE TABLE companies (
    company_id BIGINT PRIMARY KEY,
    company_name VARCHAR NOT NULL,
    industry VARCHAR
);
""")

con.execute("""
CREATE TABLE candidates (
    candidate_id BIGINT PRIMARY KEY,
    full_name VARCHAR NOT NULL,
    email VARCHAR NOT NULL,
    created_at TIMESTAMP,
    CONSTRAINT candidates_email_unique UNIQUE(email)
);
""")

con.execute("""
CREATE TABLE job_postings (
    posting_id BIGINT PRIMARY KEY,
    company_id BIGINT NOT NULL REFERENCES companies(company_id),
    title VARCHAR NOT NULL,
    location VARCHAR,
    posted_at TIMESTAMP,
    expires_at TIMESTAMP,
    status VARCHAR NOT NULL,
    created_at TIMESTAMP,
    CHECK (status IN ('draft', 'published', 'filled', 'expired')),
    CHECK (expires_at IS NULL OR posted_at IS NULL OR expires_at >= posted_at)
);
""")

con.execute("""
CREATE TABLE applications (
    application_id BIGINT PRIMARY KEY,
    posting_id BIGINT NOT NULL REFERENCES job_postings(posting_id),
    candidate_id BIGINT NOT NULL REFERENCES candidates(candidate_id),
    applied_at TIMESTAMP NOT NULL,
    stage VARCHAR NOT NULL,
    created_at TIMESTAMP,
    CHECK (stage IN (
        'submitted', 'screening', 'interview',
        'offer', 'rejected', 'withdrawn'
    ))
);
""")

# Daten mit bewusster Ordnung ein füllen
con.execute("""
INSERT INTO companies
SELECT company_id, company_name, industry
FROM stage_companies
ORDER BY company_id;
""")

con.execute("""
INSERT INTO candidates
SELECT candidate_id, full_name, email, created_at
FROM stage_candidates
ORDER BY candidate_id;
""")

con.execute("""
INSERT INTO job_postings
SELECT posting_id, company_id, title, location, posted_at, expires_at, status, created_at
FROM stage_postings
ORDER BY posted_at DESC, company_id;
""")

con.execute("""
INSERT INTO applications
SELECT application_id, posting_id, candidate_id, applied_at, stage, created_at
FROM stage_apps
ORDER BY applied_at DESC, posting_id;
""")

# Index für selektive Filter
con.execute("CREATE INDEX idx_applications_posting_id ON applications(posting_id);")

print("Final-Tabellen mit Regeln und Ordnung erstellt.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final-Tabellen mit Regeln und Ordnung erstellt.


## Schritt 6: Der I/O-Beweis
Wir führen exakt dieselbe Abfrage auf dem optimierten Schema aus. Achten Sie im Abfrageplan auf das Wort "Zone Map" oder die gelesenen Zeilen. Da DuckDB weiß, in welchen Blöcken die Firma '42' liegt, wird ein Großteil der Millionen Zeilen gar nicht erst angefasst!

In [7]:
q_designed = """
EXPLAIN ANALYZE
SELECT
    p.posting_id,
    p.status,
    COUNT(a.application_id) AS application_count
FROM job_postings p
LEFT JOIN applications a ON a.posting_id = p.posting_id
WHERE p.company_id = 42 AND p.status = 'published'
GROUP BY p.posting_id, p.status;
"""
dt_ms, plan = run_timed(q_designed)
print(f"Designed Dauer: {dt_ms:,.2f} ms")
print(plan[0][0])

Designed Dauer: 43.47 ms
analyzed_plan


## Schritt 7: Read Model erstellen


In [8]:
con.execute("DROP TABLE IF EXISTS posting_application_summary;")

con.execute("""
CREATE TABLE posting_application_summary AS
SELECT
    p.posting_id,
    p.company_id,
    p.status,
    COUNT(a.application_id) AS application_count,
    MAX(a.applied_at) AS last_application_at
FROM job_postings p
LEFT JOIN applications a ON a.posting_id = p.posting_id
GROUP BY p.posting_id, p.company_id, p.status;
""")

con.execute("CREATE INDEX idx_summary_company ON posting_application_summary(company_id);")

# Query auf Read Model
dt_ms, res = run_timed("""
SELECT company_id, SUM(application_count) AS total_apps
FROM posting_application_summary
WHERE status = 'published'
GROUP BY company_id
ORDER BY total_apps DESC
LIMIT 10;
""")

print(f"Summary query: {dt_ms:,.2f} ms; top rows: {res[:3]}")

Summary query: 5.83 ms; top rows: [(11, 15000), (12, 15000), (13, 15000)]


## Schritt 8: Hybride Daten (Text + Embeddings)


In [9]:
# Tabelle für Notizen mit Embeddings
con.execute("""
CREATE TABLE posting_notes (
    posting_id BIGINT,
    note_text VARCHAR,
    embedding FLOAT[3]
);
""")

# Demo-Daten: embedding ist hier nur ein Platzhalter
con.execute("""
INSERT INTO posting_notes
SELECT
    p.posting_id,
    'Notes for posting ' || p.posting_id::VARCHAR,
    [(p.posting_id % 7)::FLOAT, (p.posting_id % 11)::FLOAT, (p.posting_id % 13)::FLOAT]::FLOAT[3]
FROM job_postings p
LIMIT 10000;
""")

print("Hybride Daten (Text + Embeddings) erstellt.")

Hybride Daten (Text + Embeddings) erstellt.


## Schritt 9: Vektorabfrage (HNSW-Index, falls verfügbar)

In [11]:
try:
    con.execute("INSTALL vss;")
    con.execute("LOAD vss;")

    # HIER IST DER FIX: Erlaubt das Speichern von HNSW-Indizes auf der Festplatte
    con.execute("SET hnsw_enable_experimental_persistence = true;")

    # HNSW-Index erstellen
    con.execute("""
    CREATE INDEX posting_notes_hnsw
    ON posting_notes
    USING HNSW (embedding);
    """)

    # Vektorabfrage
    q_vector = """
    EXPLAIN
    SELECT *
    FROM posting_notes
    ORDER BY array_distance(embedding, [1,2,3]::FLOAT[3])
    LIMIT 5;
    """

    dt_ms, plan = run_timed(q_vector)
    print(f"Vektorabfrage Dauer: {dt_ms:,.2f} ms")
    print(plan[0][0])

except Exception as e:
    print(f"HNSW-Index nicht verfügbar oder fehlgeschlagen: {e}")
    print("Dies ist optional – der Fokus liegt auf dem Design, nicht auf der Engine.")

Vektorabfrage Dauer: 2.54 ms
physical_plan


## Schritt 10: Der Semantic Layer (Die Schnittstelle für KI)
Damit ein Text-to-SQL-Modell oder ein Agent nicht falsch joint, sperren wir die Rohtabellen weg. Wir bauen einen View (`analytics_postings`), der die komplexe Logik kapselt. Für das KI-Modell sieht das aus wie eine einfache, flache Tabelle ohne Stolperfallen.

In [12]:
con.execute("DROP VIEW IF EXISTS analytics_postings;")

con.execute("""
CREATE OR REPLACE VIEW analytics_postings AS
SELECT
    p.posting_id,
    p.company_id,
    p.title,
    p.location,
    p.status,
    p.posted_at,
    p.expires_at,
    COALESCE(s.application_count, 0) AS application_count,
    s.last_application_at
FROM job_postings p
LEFT JOIN posting_application_summary s
    ON s.posting_id = p.posting_id;
""")

# Query auf View
dt_ms, res = run_timed("""
SELECT company_id, SUM(application_count) AS total_apps
FROM analytics_postings
WHERE status = 'published'
GROUP BY company_id
ORDER BY total_apps DESC
LIMIT 10;
""")

print(f"View-Query: {dt_ms:,.2f} ms; top rows: {res[:3]}")

View-Query: 10.86 ms; top rows: [(11, 15000), (12, 15000), (13, 15000)]


## Abschluss


In [13]:
# Verbindung schließen
con.close()

print("Notebook abgeschlossen.")
print("Dieses Notebook demonstriert die im Artikel beschriebenen Prinzipien.")
print("Der Fokus liegt auf Database Design, nicht auf der zugrundeliegenden Engine.")

Notebook abgeschlossen.
Dieses Notebook demonstriert die im Artikel beschriebenen Prinzipien.
Der Fokus liegt auf Database Design, nicht auf der zugrundeliegenden Engine.
